In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# 1. قراءة البيانات
df = pd.read_excel("كل_البيانات.xlsx")
df.columns = df.columns.str.strip()
data = df.copy()

# 2. تنظيف النصوص وتوحيد المسميات
def clean_text(x):
    if pd.isna(x):
        return "غير مسجل"
    return " ".join(str(x).replace("ـ", "").split())

for col in ["المنطقة الإدارية", "الفرع", "الخدمة",
            "الجنس", "الحالة الاجتماعية", "الجنسية",
            "المستوى التعليمي", "أسباب الدخول"]:
    data[col] = data[col].apply(clean_text)

data["المنطقة الإدارية"] = (
    data["المنطقة الإدارية"]
    .str.replace(r"^(منطقة|المنطقة)\s+", "", regex=True)
    .str.replace(r"^ال+شرقية$", "الشرقية", regex=True)
)

data["أسباب الدخول"] = data["أسباب الدخول"].replace({
    "كبر سن وعدم قدره علي خدمة النفس": "كبر سن وعدم القدرة على خدمة النفس",
    "حاله نفسيه": "حالة نفسية"
})

data["الحالة الاجتماعية"] = data["الحالة الاجتماعية"].replace({
    "اعزب": "أعزب/عزباء", "أعزب/ عزباء": "أعزب/عزباء",
    "مطلق": "مطلق/مطلقة", "أرمل/ة": "أرمل/أرملة",
    "متزوج": "متزوج/متزوجة"
})

# 3. توحيد أسماء الفروع
branches = {
    "دارالرعايةبمكةالمكرمة": "دار الرعاية الاجتماعية بمكة المكرمة",
    "دارالرعاية الاجتماعيه بالمدينةالمنورة": "دار الرعاية الاجتماعية بالمدينة المنورة",
    "دارالرعايةبعنيزة": "دار الرعاية الاجتماعية بعنيزة",
    "دار الرعاية الاجتماعية بعنيزه": "دار الرعاية الاجتماعية بعنيزة",
    "دارالرعايةبابها": "دار الرعاية الاجتماعية بأبها",
    "دارالرعايةالاجتماعيةبالدمام": "دار الرعاية الاجتماعية بالدمام",
    "دارالرعاية الاجتماعيه بالرياض": "دار الرعاية الاجتماعية للمسنين بالرياض",
    "دار رعاية المسنات بالرياض": "دار الرعاية الاجتماعية للمسنات بالرياض",
    "دارالرعايةالاجتماعية بالطائف": "دار الرعاية الاجتماعية بالطائف"
}
data["الفرع"] = data["الفرع"].replace(branches)

# 4. تصنيف المستوى التعليمي
def education(x):
    if x == "غير مسجل" or x == "غير محدد":
        return "غير محدد"
    if "متعلم" in x and "غير محدد" in x:
        return "متعلم - المستوى غير محدد"
    if "أمي" in x:
        return "أمي"
    if "يقراء ويكتب" in x or "يقرأ ويكتب" in x:
        return "يقرأ ويكتب"
    if "ابتدائ" in x or "أبتدائ" in x:
        return "ابتدائي"
    if "فوق المتوسط" in x:
        return "فوق المتوسط"
    if "متوسط" in x:
        return "متوسط"
    if "ثانو" in x:
        return "ثانوي"
    if "دبلوم" in x:
        return "دبلوم"
    if "بكالور" in x or "جامعية" in x:
        return "جامعي"
    if x == "يقرأ":
        return "يقرأ"
    return "غير مصنف"

data["الفئة التعليمية"] = data["المستوى التعليمي"].apply(education)

# 5. تصنيف خدمات الرعاية
services = {
    "رعايه اجتماعيه للمسنين": "رعاية اجتماعية للمسنين",
    "الرعاية الايوائية داخل دور الرعاية الاجتماعية": "رعاية إيوائية في دور الرعاية الاجتماعية",
    "اسناد خدمات الرعاية الايوائية في مراكز كبار السن الأهلية التابعة للقطاع غير الربحي": "إسناد الرعاية الإيوائية للقطاع غير الربحي",
    "صرف بطاقة تخفيض الاركاب": "خدمة غير إيوائية"
}

services = {clean_text(k): v for k, v in services.items()}
data["نوع الخدمة"] = data["الخدمة"].map(services)

if data["نوع الخدمة"].isna().any():
    print("خدمات تحتاج مراجعة:")
    display(data.loc[data["نوع الخدمة"].isna(), "الخدمة"].value_counts())

df_analysis = data[
    data["نوع الخدمة"].isin([
        "رعاية اجتماعية للمسنين",
        "رعاية إيوائية في دور الرعاية الاجتماعية",
        "إسناد الرعاية الإيوائية للقطاع غير الربحي"
    ])
].copy()

df_analysis["نوع_الرعاية"] = np.where(
    df_analysis["نوع الخدمة"].str.contains("غير الربحي"),
    "إسناد للقطاع غير الربحي",
    "حكومي"
)

# 6. تجهيز السنة والفئات العمرية
df_analysis["السنة"] = df_analysis["السنة"].astype(str).str.strip()
df_analysis["السنة"] = df_analysis["السنة"].apply(
    lambda x: "2026 Q2" if "2026" in x else x
)

df_analysis["العمر"] = pd.to_numeric(
    df_analysis["العمر"], errors="coerce"
)

df_analysis["الفئة العمرية"] = pd.cut(
    df_analysis["العمر"],
    bins=[-np.inf, 59, 69, 79, 89, np.inf],
    labels=["Under 60", "60-69", "70-79", "80-89", "90+"]
)


In [2]:
# 7. المؤشرات والتحليلات
periods = ["2021", "2024", "2025", "2026 Q2"]
care_types = ["حكومي", "إسناد للقطاع غير الربحي"]

yearly = pd.crosstab(
    df_analysis["السنة"], df_analysis["نوع_الرعاية"]
).reindex(index=periods, columns=care_types, fill_value=0)

yearly["الإجمالي"] = yearly[care_types].sum(axis=1)
yearly["نسبة الإسناد %"] = (
    yearly[care_types[1]] / yearly["الإجمالي"].replace(0, np.nan) * 100
).round(1)

geo = pd.crosstab(
    df_analysis["المنطقة الإدارية"], df_analysis["نوع_الرعاية"]
).reindex(columns=care_types, fill_value=0)

centers = df_analysis.loc[
    df_analysis["نوع_الرعاية"] == care_types[1], "الفرع"
].value_counts().rename("عدد السجلات").to_frame()

profiles = {
    "Age": pd.crosstab(df_analysis["نوع_الرعاية"], df_analysis["الفئة العمرية"]),
    "Gender": pd.crosstab(df_analysis["نوع_الرعاية"], df_analysis["الجنس"]),
    "Marital": pd.crosstab(df_analysis["نوع_الرعاية"], df_analysis["الحالة الاجتماعية"]),
    "Education": pd.crosstab(df_analysis["نوع_الرعاية"], df_analysis["الفئة التعليمية"]),
    "Reasons": pd.crosstab(df_analysis["نوع_الرعاية"], df_analysis["أسباب الدخول"])
}

coverage = pd.crosstab(
    df_analysis["المنطقة الإدارية"], df_analysis["السنة"]
).reindex(columns=periods, fill_value=0)

kpis = pd.DataFrame({
    "المؤشر": ["إجمالي السجلات", "حكومي", "إسناد غير ربحي",
               "نسبة الإسناد %", "متوسط عمر الإسناد"],
    "القيمة": [
        len(df_analysis),
        (df_analysis["نوع_الرعاية"] == care_types[0]).sum(),
        (df_analysis["نوع_الرعاية"] == care_types[1]).sum(),
        round((df_analysis["نوع_الرعاية"] == care_types[1]).mean() * 100, 1),
        round(df_analysis.loc[
            df_analysis["نوع_الرعاية"] == care_types[1], "العمر"
        ].mean(), 1)
    ]
})

In [3]:
# 8. التحقق من جودة النتائج
print("السجلات الأصلية:", len(df))
print("سجلات الإيواء:", len(df_analysis))
print("السجلات المتطابقة:", df_analysis.duplicated(subset=df.columns.tolist()).sum())
print("الأعمار أقل من 60:", (df_analysis["العمر"] < 60).sum())
print("خدمات غير مصنفة:", data["نوع الخدمة"].isna().sum())

assert yearly["الإجمالي"].sum() == len(df_analysis)
assert geo.to_numpy().sum() == len(df_analysis)

print("\nمؤشرات الأداء:")
display(kpis)
print("\nالتوزيع حسب السنوات:")
display(yearly)
print("\nالتوزيع حسب المناطق:")
display(geo)



السجلات الأصلية: 1283
سجلات الإيواء: 1282
السجلات المتطابقة: 136
الأعمار أقل من 60: 136
خدمات غير مصنفة: 0

مؤشرات الأداء:


,المؤشر,القيمة
0,إجمالي السجلات,1282.0
1,حكومي,935.0
2,إسناد غير ربحي,347.0
3,نسبة الإسناد %,27.1
4,متوسط عمر الإسناد,77.0



التوزيع حسب السنوات:


نوع_الرعاية,حكومي,إسناد للقطاع غير الربحي,الإجمالي,نسبة الإسناد %
السنة,,,,
2021,414,0,414,0.0
2024,350,26,376,6.9
2025,85,163,248,65.7
2026 Q2,86,158,244,64.8



التوزيع حسب المناطق:


نوع_الرعاية,حكومي,إسناد للقطاع غير الربحي
المنطقة الإدارية,,
الباحة,5,175
الجوف,6,0
الحدود الشمالية,2,0
الرياض,347,0
الشرقية,71,59
القصيم,95,58
المدينة المنورة,101,55
تبوك,5,0
جازان,20,0


In [6]:
# 9. حفظ النتائج مرة واحدة
output = Path(r"C:\Users\Basmah\STC")
output.mkdir(exist_ok=True)

df_analysis.to_csv(
    output / "FinalData.csv",
    index=False, encoding="utf-8-sig"
)

with pd.ExcelWriter(output / "Bawsala_Final_Insights.xlsx") as writer:
    kpis.to_excel(writer, sheet_name="KPIs", index=False)
    yearly.to_excel(writer, sheet_name="Yearly")
    geo.to_excel(writer, sheet_name="Geography")
    centers.to_excel(writer, sheet_name="Centers")
    coverage.to_excel(writer, sheet_name="Coverage")
    for name, table in profiles.items():
        table.to_excel(writer, sheet_name=name)

print("\nتم حفظ النتائج في:", output.resolve())



تم حفظ النتائج في: C:\Users\Basmah\STC


In [5]:
df_analysis.to_csv(
    output / "FinalData.csv",
    index=False,
    encoding="utf-8-sig"
)